# 🗞️ Bias Detector — GUI Interface

This notebook loads the already-trained model from Google Drive and launches the interactive GUI.
**No training required.** Just run all cells top to bottom.

> Make sure you have already run the training notebook and saved the model to Drive.

In [ ]:
# ============================================================
# CELL 1: Install dependencies — run once, then restart runtime
# ============================================================
!pip install gradio --quiet
!pip install transformers --quiet
print("Done. Restart runtime if this is your first run.")

In [ ]:
# ============================================================
# CELL 2: Mount Drive + load model
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import re
from bs4 import BeautifulSoup
import unicodedata

SAVE_PATH    = '/content/drive/MyDrive/Colab_Notebooks/bias_detector_roberta'
TORCH_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_LEN      = 256
id2label     = {0: "biased", 1: "non-biased"}

print(f"Loading model from {SAVE_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH).to(TORCH_DEVICE)
model.eval()
print(f"Model loaded on {TORCH_DEVICE}. Ready.")

In [ ]:
# ============================================================
# CELL 3: Prediction helper
# ============================================================
_RE_MULTI_SPACE = re.compile(r'\s+')

def light_clean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = BeautifulSoup(text, 'lxml').get_text()
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    return _RE_MULTI_SPACE.sub(' ', text).strip()

def predict_bias(text: str, threshold: float = 0.5) -> dict:
    """
    Predict bias for a headline or sentence.
    Returns: { label, confidence, scores }
    """
    cleaned = light_clean(text)
    inputs  = tokenizer(
        cleaned,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LEN,
        padding=True
    ).to(TORCH_DEVICE)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs      = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
    pred_id    = int(probs.argmax())
    confidence = float(probs[pred_id])
    label      = id2label[pred_id] if confidence >= threshold else "uncertain"

    return {
        "label":      label,
        "confidence": round(confidence, 4),
        "scores":     {id2label[i]: round(float(p), 4) for i, p in enumerate(probs)}
    }

# Smoke test
_test = predict_bias("The committee voted 7-3 to pass the bill.")
print(f"Smoke test → {_test['label']} ({_test['confidence']:.1%})")

In [ ]:
# ============================================================
# CELL 4: Launch GUI
# A public gradio.live link will appear below — open it in any browser.
# ============================================================
import gradio as gr

def confidence_bar(score: float) -> str:
    filled = round(score * 20)
    return "█" * filled + "░" * (20 - filled) + f"  {score:.1%}"

def analyse(headline: str):
    if not headline or not headline.strip():
        return "—", "Enter a headline above.", gr.update(visible=False)

    result     = predict_bias(headline.strip())
    label      = result["label"]
    confidence = result["confidence"]
    scores     = result["scores"]

    if label == "biased":
        verdict = f"⚠️  BIASED  ({confidence:.1%} confidence)"
        tip     = "💡 **Tip:** Biased headlines often use emotionally charged words, generalisations, or framing that implies a conclusion. Try rewriting with neutral, factual language."
    elif label == "non-biased":
        verdict = f"✅  NON-BIASED  ({confidence:.1%} confidence)"
        tip     = "💡 **Tip:** This headline reads as factual and neutral. It avoids emotional language and presents information without implied judgement."
    else:
        verdict = f"🔍  UNCERTAIN  ({confidence:.1%} confidence)"
        tip     = "💡 **Tip:** The model isn't confident. Try rephrasing or providing more context."

    breakdown = ""
    for lbl, score in scores.items():
        icon = "🔴" if lbl == "biased" else "🟢"
        breakdown += f"{icon}  {lbl.upper():<14}{confidence_bar(score)}\n"

    return verdict, breakdown, gr.update(value=tip, visible=True)


CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@700&family=Source+Serif+4:ital,wght@0,300;0,400;1,300&display=swap');

body, .gradio-container {
    background: #f5f0e8 !important;
    font-family: 'Source Serif 4', Georgia, serif !important;
}
h1 { font-family: 'Playfair Display', serif !important; font-size: 2.2rem !important;
     color: #1a1a2e !important; letter-spacing: -0.5px !important; }
.gr-button-primary {
    background: #1a1a2e !important; color: #f5f0e8 !important;
    border: none !important; font-family: 'Source Serif 4', serif !important;
    font-size: 1rem !important; border-radius: 2px !important;
    padding: 12px 28px !important; transition: background 0.2s !important;
}
.gr-button-primary:hover { background: #c0392b !important; }
.gr-button-secondary {
    background: transparent !important; color: #1a1a2e !important;
    border: 1px solid #1a1a2e !important; border-radius: 2px !important;
}
textarea, input[type=text] {
    font-family: 'Source Serif 4', serif !important; font-size: 1.1rem !important;
    border: 1px solid #c8bfb0 !important; border-radius: 2px !important;
    background: #fffdf8 !important; color: #1a1a2e !important;
}
label span {
    font-family: 'Source Serif 4', serif !important; font-weight: 400 !important;
    color: #4a4540 !important; font-size: 0.85rem !important;
    text-transform: uppercase !important; letter-spacing: 1px !important;
}
"""

EXAMPLES = [
    ["The radical left is destroying everything hardworking Americans have built."],
    ["The Senate voted 52-48 to pass the infrastructure bill on Thursday."],
    ["Scientists release new data on rising global temperatures."],
    ["The corrupt administration once again betrayed the people who trusted it."],
    ["Apple reported quarterly earnings of $94.8 billion, up 5% year-over-year."],
    ["The out-of-touch elite push their dangerous agenda on ordinary families."],
]

with gr.Blocks(css=CSS, title="News Bias Detector") as demo:

    gr.HTML("""
        <div style='padding:32px 0 8px 0; border-bottom:3px double #1a1a2e; margin-bottom:24px;'>
            <div style='font-size:0.75rem; letter-spacing:3px; text-transform:uppercase;
                        color:#6b6560; font-family:Source Serif 4,serif; margin-bottom:8px;'>
                Media Analysis Tool
            </div>
            <h1>News Bias Detector</h1>
            <div style='font-style:italic; color:#6b6560; font-size:1rem; margin-top:4px;
                        font-family:Source Serif 4,serif;'>
                Fine-tuned RoBERTa · 82% accuracy · Trained on BABE dataset
            </div>
        </div>
    """)

    with gr.Row():
        with gr.Column(scale=3):
            headline_input = gr.Textbox(
                label="Headline or Sentence",
                placeholder="Paste a news headline here…",
                lines=3,
                max_lines=6,
            )
            with gr.Row():
                analyse_btn = gr.Button("Analyse →", variant="primary")
                clear_btn   = gr.Button("Clear", variant="secondary")

            gr.Examples(
                examples=EXAMPLES,
                inputs=headline_input,
                label="Try an example",
            )

        with gr.Column(scale=2):
            verdict_out   = gr.Textbox(label="Verdict", interactive=False)
            breakdown_out = gr.Textbox(label="Confidence Scores", interactive=False, lines=4)
            tip_out       = gr.Markdown(visible=False)

    gr.HTML("""
        <div style='margin-top:32px; padding-top:16px; border-top:1px solid #c8bfb0;
                    font-family:Source Serif 4,serif; font-style:italic;
                    color:#9a9088; font-size:0.85rem; text-align:center;'>
            Predictions are probabilistic. Model accuracy: 82% on held-out test set.
        </div>
    """)

    analyse_btn.click(
        fn=analyse,
        inputs=headline_input,
        outputs=[verdict_out, breakdown_out, tip_out],
    )
    clear_btn.click(
        fn=lambda: ("", "", "", gr.update(visible=False)),
        outputs=[headline_input, verdict_out, breakdown_out, tip_out],
    )

demo.launch(share=True)